In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import zipfile
import io

In [ ]:
# Download MyAnimeList data from GitHub
url = 'https://github.com/Hernan4444/MyAnimeList-Database/archive/refs/heads/master.zip'
r = requests.get(url, stream=True)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall()

# Load the anime data
anime_data_raw = pd.read_csv('MyAnimeList-Database-master/data/anime.csv')
print(f"Raw dataset shape: {anime_data_raw.shape}")
anime_data_raw.head()

Raw dataset shape: (17562, 35)


,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [ ]:
# Preprocessing — Do NOT modify this cell
columns_to_keep = ['MAL_ID', 'Name', 'Score', 'Type', 'Episodes',
                   'Members', 'Completed', 'Watching', 'Dropped', 'Popularity']
anime_data = anime_data_raw[columns_to_keep].copy()

# Remove rows where key metric columns have missing or zero values
anime_data = anime_data.dropna(subset=['Members', 'Completed', 'Watching'])
anime_data = anime_data[(anime_data['Members'] > 0) &
                        (anime_data['Completed'] > 0) &
                        (anime_data['Watching'] > 0)]
anime_data = anime_data.reset_index(drop=True)
print(f"Dataset shape after preprocessing: {anime_data.shape}")
anime_data.head()

Dataset shape after preprocessing: (16943, 10)


,MAL_ID,Name,Score,Type,Episodes,Members,Completed,Watching,Dropped,Popularity
0,1,Cowboy Bebop,8.78,TV,26,1251960,718161,105808,26678,39
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,Movie,1,273145,208333,4143,770,518
2,6,Trigun,8.24,TV,26,558913,343492,29113,13925,201
3,7,Witch Hunter Robin,7.27,TV,26,94683,46165,4300,5378,1467
4,8,Bouken Ou Beet,6.98,TV,52,13224,7314,642,1108,4369


In [2]:
def homework(anime_data, metric_column, n):
    # Sort the anime in descending order based on the values in the column specified by metric_column
    sorted_anime = anime_data.sort_values(by=metric_column, ascending=False).reset_index(drop=True)

    # Group Anime into n Equal Parts using pd.qcut() on the rank of the metric values
    # First, calculate the rank of the metric_column values
    sorted_anime['rank'] = sorted_anime[metric_column].rank(method='first', ascending=False)

    # Use pd.qcut to divide the anime into n groups based on their ranks
    # Labels are created from 1 to n to represent the groups
    sorted_anime['group'] = pd.qcut(sorted_anime['rank'], q=n, labels=False, duplicates='drop') + 1

    # Calculate Proportion for Each Group
    # Calculate the total of the metric_column for each group
    group_totals = sorted_anime.groupby('group')[metric_column].sum().reset_index()
    group_totals = group_totals.rename(columns={metric_column: 'group_metric_total'})

    # Calculate the overall total of the metric_column
    overall_metric_total = sorted_anime[metric_column].sum()

    # Calculate the percentage for each group (now summing to 1.0)
    group_totals['proportion'] = (group_totals['group_metric_total'] / overall_metric_total)

    # Sort Groups and Label
    # Sort the groups in descending order of their proportion
    group_totals = group_totals.sort_values(by='proportion', ascending=False).reset_index(drop=True)

    # Label them as "Group 1", "Group 2", ..., "Group n"
    group_totals['group_label'] = ['Group ' + str(i + 1) for i in range(len(group_totals))]

    # Return the result as a pandas.Series with labels as index and proportion as values
    my_result = group_totals.set_index('group_label')['proportion']

    return my_result

In [3]:
def homework(anime_data, metric_column, n):

    sorted_anime = anime_data.sort_values(by=metric_column, ascending=False).reset_index(drop=True)


    sorted_anime['rank'] = sorted_anime[metric_column].rank(method='first', ascending=False)


    sorted_anime['group'] = pd.qcut(sorted_anime['rank'], q=n, labels=False, duplicates='drop') + 1


    group_totals = sorted_anime.groupby('group')[metric_column].sum().reset_index()
    group_totals = group_totals.rename(columns={metric_column: 'group_metric_total'})

    overall_metric_total = sorted_anime[metric_column].sum()


    group_totals['proportion'] = (group_totals['group_metric_total'] / overall_metric_total)


    group_totals = group_totals.sort_values(by='proportion', ascending=False).reset_index(drop=True)


    group_totals['group_label'] = ['Group ' + str(i + 1) for i in range(len(group_totals))]


    my_result = group_totals.set_index('group_label')['proportion']

    return my_result

In [ ]:
def homework(anime_data, metric_column, n):

    sorted_anime = anime_data.sort_values(by=metric_column, ascending=False).reset_index(drop=True)


    sorted_anime['rank'] = sorted_anime[metric_column].rank(method='first', ascending=False)


    sorted_anime['group'] = pd.qcut(sorted_anime['rank'], q=n, labels=False, duplicates='drop') + 1


    group_totals = sorted_anime.groupby('group')[metric_column].sum().reset_index()
    group_totals = group_totals.rename(columns={metric_column: 'group_metric_total'})


    overall_metric_total = sorted_anime[metric_column].sum()

    group_totals['proportion'] = (group_totals['group_metric_total'] / overall_metric_total) * 100


    group_totals = group_totals.sort_values(by='proportion', ascending=False).reset_index(drop=True)


    group_totals['group_label'] = ['Group ' + str(i + 1) for i in range(len(group_totals))]


    my_result = group_totals.set_index('group_label')['proportion']

    return my_result

In [ ]:
import ast
import inspect
import textwrap

def hw4_public_tests(homework_func):
    print("=== HW4 Public Tests ===")

    # Get the source code of the homework function
    source = textwrap.dedent(inspect.getsource(homework_func))
    tree = ast.parse(source)

    # ---------- Test 1: No import statements ----------
    print("Test 1: No import statements in function...", end=" ")
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            print("NG")
            if isinstance(node, ast.Import):
                names = ", ".join(alias.name for alias in node.names)
            else:
                names = node.module
            print(f"  Found import statement: '{names}'")
            print("  Remove all import statements before submitting.")
            return
    print("OK")

    # ---------- Test 2: No data loading code ----------
    print("Test 2: No data loading code in function...", end=" ")
    blocked_calls = {"read_csv", "read_excel", "read_json", "read_html",
                     "get", "post", "ZipFile", "extractall", "urlopen"}
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            func_name = ""
            if isinstance(node.func, ast.Attribute):
                func_name = node.func.attr
            elif isinstance(node.func, ast.Name):
                func_name = node.func.id
            if func_name in blocked_calls:
                print("NG")
                print(f"  Found data loading call: '{func_name}()'")
                print("  Do not include data downloading/loading code in your function.")
                return
    print("OK")

    # ---------- Test 3: No hardcoded file paths or URLs ----------
    print("Test 3: No hardcoded file paths or URLs...", end=" ")
    for node in ast.walk(tree):
        if isinstance(node, ast.Constant) and isinstance(node.value, str):
            val = node.value
            if val.startswith("http") or ".csv" in val or ".zip" in val:
                print("NG")
                print(f"  Found hardcoded path/URL: '{val[:60]}...'")
                print("  Do not include file paths or URLs in your function.")
                return
    print("OK")

    print("=== All Public Tests Passed ===")

hw4_public_tests(homework)

=== HW4 Public Tests ===
Test 1: No import statements in function... OK
Test 2: No data loading code in function... OK
Test 3: No hardcoded file paths or URLs... OK
=== All Public Tests Passed ===
